## Why Ingress & Ingress controllers

You have Services — L4 inside, or public via `type: LoadBalancer`. So why isn't that the whole answer? **One LoadBalancer per Service is expensive and rigid:** each provisions a separate cloud LB (tens of dollars/month, a separate public IP), with **no HTTP-level routing** (can't split `/api/*` vs `/static/*`), no TLS termination, no host-based sharing of one IP.

What you want is **one public entry point** in front of many Services, HTTP-aware:

```
api.example.com      →  Ingress    →  Service "api"  → pods
www.example.com/api* →  controller →  Service "api"  → pods
www.example.com/*    →  (one LB IP)→  Service "web"  → pods
```

**Ingress is the *resource*; the controller does the work.** An `Ingress` object is declarative config that does nothing alone — you install an **ingress controller** that watches Ingress resources and configures the real proxy. Without a controller, an Ingress is just YAML in `etcd`.

Controllers: **`ingress-nginx`** (most deployed, CNCF — not the commercial Nginx), **Traefik**, **HAProxy**, **AWS Load Balancer Controller** (ALB on EKS), **GKE Ingress**, Istio/Contour (Envoy). Pick **one per cluster**, install via Helm.

**`IngressClass`** routes an Ingress to a controller: an Ingress sets `spec.ingressClassName: nginx`; one class can be default (via annotation). The controller is itself a workload, fronted by a `LoadBalancer` Service (one LB per *controller*, not per Ingress — the whole cost saving), a `NodePort`, or `hostNetwork`. `kind` ships none; install `ingress-nginx` to experiment. On our map this is the **Ingress** chip in the Networking box — the L7 door in front of many Services.